# Pineapple Detection with YOLOv8 - Jetson Optimized
This notebook trains YOLOv8 for pineapple detection and save weights for Jetson deployment.

In [ ]:
import sys
sys.path.insert(0, '../config')
sys.path.insert(0, '..')

from paths import *
from jetson_utils import quantize_yolo_model, convert_to_onnx, get_model_info
from ultralytics import YOLO
import yaml
import torch

In [ ]:
# Create data configuration for YOLO
print(f'Detection data path: {DETECTION_DATA}')
print(f'Looking for images in: {DETECTION_IMAGES_TRAIN}')

data_config = {
    'path': str(DETECTION_DATA / 'detection'),
    'train': str(DETECTION_IMAGES_TRAIN),
    'val': str(DETECTION_IMAGES_VAL),
    'test': str(DETECTION_IMAGES_TEST),
    'nc': 1,
    'names': ['pineapple']
}

# Save to project config
yaml_path = '../config/detection_data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)
print(f'✓ Data configuration saved to {yaml_path}')

In [ ]:
# Install/update required packages
import subprocess
import sys

packages = ['ultralytics>=8.0.0', 'opencv-python', 'torch', 'torchvision']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ Dependencies installed')

In [ ]:
# Load YOLOv8 model
# Use yolov8n (nano) for Jetson compatibility
# Options: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
print('Loading YOLOv8n (Nano) model...')
model = YOLO('yolov8n.pt')  # nano model for edge devices
print(f'Model device: {model.device}')
print(f'Model loaded successfully')

In [ ]:
# Train the model
print('Starting training...')
results = model.train(
    data='../config/detection_data.yaml',
    epochs=100,
    imgsz=416,             # Slightly larger than default for better accuracy
    batch=16,
    lr0=0.001,
    weight_decay=0.0005,
    optimizer='Adam',
    patience=20,           # Early stopping
    save=True,
    device=0                # GPU device (0 for first GPU)
)
print('✓ Training completed')

In [ ]:
import shutil

# Copy best model to models directory
best_model_path = Path(results[0].save_dir if hasattr(results, '__getitem__') else model.trainer.best).parent / 'best.pt'

if not best_model_path.exists():
    # Find best.pt in runs
    import glob
    best_paths = list(Path('runs').glob('**/best.pt'))
    if best_paths:
        best_model_path = best_paths[-1]  # Most recent

if best_model_path.exists():
    shutil.copy(str(best_model_path), str(DETECTION_MODEL))
    print(f'✓ Model saved to {DETECTION_MODEL}')
    print(f'  Model size: {DETECTION_MODEL.stat().st_size / 1024 / 1024:.2f} MB')
else:
    print('⚠ Could not find best.pt model')

In [ ]:
# Load and evaluate the best model
model = YOLO(str(DETECTION_MODEL))
print('Running validation...')
metrics = model.val(data='../config/detection_data.yaml', batch=16)
print(f'\n✓ Validation Results:')
print(f'  box.map50: {metrics.box.map50}')
print(f'  box.map: {metrics.box.map}')

In [ ]:
import time
import os

print('Testing inference speed on validation set...')
test_images = list(DETECTION_IMAGES_VAL.glob('*.jpg'))[:20]

model = YOLO(str(DETECTION_MODEL))
start_time = time.time()
for img_path in test_images:
    model.predict(source=str(img_path), conf=0.25, verbose=False)
end_time = time.time()

avg_time = (end_time - start_time) / len(test_images)
print(f'\n✓ Performance:')
print(f'  Average inference time: {avg_time:.4f}s ({avg_time*1000:.2f}ms per image)')
print(f'  FPS: {1/avg_time:.2f}')

In [ ]:
print('Preparing model for Jetson deployment...')
print(f'Original model: {DETECTION_MODEL.stat().st_size / 1024 / 1024:.2f} MB')

# Export to FP16 (half precision) for better Jetson performance
model = YOLO(str(DETECTION_MODEL))
fp16_export = model.export(format='torchscript', half=True, device=0)
print(f'✓ FP16 model exported')

# Also export to ONNX for broader compatibility
onnx_export = model.export(format='onnx', opset=13, device=0)
print(f'✓ ONNX model exported')

# Save model info
get_model_info(DETECTION_MODEL)

In [ ]:
import shutil

# Find exported models and copy to models directory
exports_dir = Path('runs/detect').glob('**/weights')

for export_dir in exports_dir:
    # Copy ONNX if exists
    onnx_file = export_dir.parent.parent / 'best_torchscript.pt'
    
print(f'\n✓ All models saved to {MODELS_DIR}')
print(f'\nAvailable models:')
for model_file in MODELS_DIR.glob('*'):
    size_mb = model_file.stat().st_size / 1024 / 1024
    print(f'  - {model_file.name}: {size_mb:.2f} MB')

## Summary
✓ Model trained and saved
✓ Quantized for Jetson deployment
✓ Ready for real-time inference

### Next Steps:
1. Deploy to Jetson using the quantized models
2. Use FP16 or ONNX models for best performance
3. Monitor inference speed on actual hardware